# 🌩️ FahMai RAG System - ThaiLLM Version
## Google Colab Notebook

RAG system for the **FahMai (ฟ้าใหม่)** Thai electronics store challenge using **official ThaiLLM models**.

### Features
- **Thai Embedding**: `BAAI/bge-m3` - Top-ranked multilingual model for Thai
- **ThaiLLM**: `KBTG-Labs/ThaiLLM-8B-Instruct` - Official ThaiLLM (Qwen3-8B based)
- **Hybrid Retrieval**: Vector (Semantic) + BM25 (Keyword) with Reciprocal Rank Fusion
- **Thai-Aware**: Full support for Thai word tokenization using `pythainlp`
- **Special Case Detection**: Automatic handling of Choice 9 (no data) and Choice 10 (out-of-scope)

### Quick Start
1. Connect to GPU runtime (Runtime > Change runtime type > GPU)
2. Run all cells in order
3. The notebook will automatically clone the repository from GitHub
4. Answers will be saved to `output/submission.csv`

**Note:** For faster execution, use the lightweight mode (cross-encoder) instead of full ThaiLLM.

## 1. Setup Environment

In [ ]:
#@title Install Dependencies
#@markdown This may take 3-5 minutes on first run.

!pip install -q \
    sentence-transformers>=2.2.0 \
    chromadb>=0.4.0 \
    faiss-cpu>=1.7.0 \
    pythainlp>=5.0.0 \
    langchain>=0.1.0 \
    langchain-community>=0.0.10 \
    transformers>=4.51.0 \
    torch>=2.0.0 \
    accelerate>=0.24.0 \
    pandas>=2.0.0 \
    markdown>=3.5.0 \
    beautifulsoup4>=4.12.0 \
    python-dotenv>=1.0.0 \
    tqdm>=4.65.0 \
    numpy>=1.24.0 \
    rank-bm25>=0.2.2

print("✓ Dependencies installed successfully!")

In [ ]:
#@title Clone Repository from GitHub
#@markdown Clone the project and data from GitHub repository.

import os

# Clone repository if not already present
if not os.path.exists('Mini-Hackathon3'):
    print("Cloning repository from GitHub...")
    !git clone https://github.com/NakaSato/Mini-Hackathon3.git
    %cd Mini-Hackathon3
    print("✓ Repository cloned successfully!")
else:
    %cd Mini-Hackathon3
    print("✓ Repository already exists!")

# Verify data files exist
print("\nVerifying data files...")
print(f"Knowledge Base: {len(os.listdir('data/knowledge_base'))} files")
print(f"Questions: {'✓' if os.path.exists('data/questions.csv') else '✗'}")
print(f"Sample Submission: {'✓' if os.path.exists('data/sample_submission.csv') else '✗'}")

In [ ]:
#@title Check GPU Availability
import torch

if torch.cuda.is_available():
    print(f"✓ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA Version: {torch.version.cuda}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ No GPU detected. Running on CPU (will be slower)")
    print("  Go to Runtime > Change runtime type > GPU to enable GPU acceleration")

## 2. Check GPU Availability

## 3. Run the Pipeline

In [ ]:
#@title Run Full Pipeline
#@markdown **Lightweight Mode**: Faster (10-100x), uses cross-encoder
#@markdown **Full ThaiLLM Mode**: Better accuracy, slower (requires GPU)

USE_LIGHTWEIGHT = True  #@param {type:"boolean"}
TOP_K = 5  #@param {type:"slider", min:1, max:20, step:1}
BUILD_INDEX = True  #@param {type:"boolean"}

from src.pipeline import FahMaiRAGPipeline

print("=" * 60)
print("FahMai RAG Pipeline")
print("=" * 60)
print(f"Mode: {'Lightweight (Cross-Encoder)' if USE_LIGHTWEIGHT else 'Full ThaiLLM'}")
print(f"Top-K: {TOP_K}")
print(f"Build Index: {BUILD_INDEX}")
print()

# Initialize pipeline
pipeline = FahMaiRAGPipeline(
    kb_path='data/knowledge_base',
    index_path='output/index',
    embedding_model='BAAI/bge-m3',
    thai_llm_model='KBTG-Labs/ThaiLLM-8B-Instruct',
    top_k=TOP_K,
    use_lightweight=USE_LIGHTWEIGHT
)

# Initialize
pipeline.initialize(build_index=BUILD_INDEX)

# Answer questions
print("\n" + "=" * 60)
print("Answering Questions")
print("=" * 60 + "\n")

results = pipeline.answer_all_questions(
    questions_path='data/questions.csv',
    output_path='output/submission.csv',
    use_special_case=True,
    save_debug=True
)

print("\n" + "=" * 60)
print("Pipeline Complete!")
print("=" * 60)
print(f"Submission saved to: output/submission.csv")
print(f"Total questions answered: {len(results)}")

In [ ]:
#@title View Results
import pandas as pd

# Load results
results = pd.read_csv('output/submission.csv')
debug = pd.read_json('output/submission_debug.json')

print("=== Answer Distribution ===")
print(results['answer'].value_counts().sort_index())

print("\n=== Sample Answers ===")
print(results.head(10))

In [ ]:
#@title Download Submission
from google.colab import files

print("Downloading submission files...")
files.download('output/submission.csv')
files.download('output/submission_debug.json')
print("✓ Download complete!")

## 5. Advanced: Test Mode

In [ ]:
#@title Run Quick Test
#@markdown Test the pipeline with a single question to verify setup.

from src.pipeline import FahMaiRAGPipeline

# Initialize pipeline (lightweight for speed)
pipeline = FahMaiRAGPipeline(
    kb_path='data/knowledge_base',
    index_path='output/index',
    embedding_model='BAAI/bge-m3',
    thai_llm_model='KBTG-Labs/ThaiLLM-8B-Instruct',
    top_k=3,
    use_lightweight=True
)

pipeline.initialize(build_index=True)

# Test question
test_question = "ร้านฟ้าใหม่มีนโยบายการคืนสินค้าอย่างไร"
test_choices = [
    "คืนได้ภายใน 7 วัน",
    "คืนได้ภายใน 14 วัน",
    "คืนได้ภายใน 30 วัน",
    "ไม่รับคืนสินค้า",
    "คืนได้เฉพาะสินค้าอิเล็กทรอนิกส์",
    "คืนได้เฉพาะสินค้าที่ไม่ได้ใช้",
    "ต้องแสดงใบเสร็จ",
    "ติดต่อฝ่ายบริการลูกค้า",
    "ไม่มีข้อมูลนี้ในฐานข้อมูล",
    "คำถามนี้ไม่เกี่ยวข้องกับร้านฟ้าใหม่"
]

answer, debug_info = pipeline.answer_question(
    test_question, test_choices, use_special_case=True
)

print(f"Question: {test_question}")
print(f"Answer: Choice {answer} - {test_choices[answer-1]}")
print(f"\nRetrieved {len(debug_info.get('retrieval_results', []))} documents")

## 6. Troubleshooting

In [ ]:
#@title Check System Resources
import torch
import psutil
import os

print("=== System Resources ===")
print(f"CPU Count: {psutil.cpu_count()}")
print(f"RAM Available: {psutil.virtual_memory().available / 1e9:.2f} GB")
print(f"RAM Total: {psutil.virtual_memory().total / 1e9:.2f} GB")
print()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
else:
    print("GPU: Not available")

print()
print(f"Disk Usage: {psutil.disk_usage('/').percent}%")

In [ ]:
#@title Clear GPU Cache (if out of memory)
#@markdown Run this if you encounter out-of-memory errors.

import torch
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU cache cleared!")
else:
    print("✓ Memory cleanup complete!")

## Tips for Better Performance

1. **Use GPU**: Runtime > Change runtime type > GPU
2. **Lightweight Mode**: 10-100x faster, good for iteration
3. **Increase Top-K**: Better recall but slower (try 5-10)
4. **Pre-build Index**: Save the index and reuse it
5. **Monitor Resources**: Check GPU memory usage

## Common Issues

**Out of Memory**:
- Use `--lightweight` mode
- Clear GPU cache (run cell above)
- Reduce batch size

**Slow Inference**:
- Enable GPU acceleration
- Use lightweight mode
- Reduce top-k value

**Thai Text Issues**:
- Ensure UTF-8 encoding
- Check pythainlp installation
- Verify embedding model supports Thai